<p align="center"><img src="../Additionals/Empa-Workshops-Template-Banner.jpg" alt="University Workshops" style="display: block; margin: 0 auto" height=/></p>

## Welcome to the Accelerator Workshops!

Welcome to the Edge AI step of our Accelerator Workshops series organized by Empa Electronics. This open-source repository contains the working environment for the "Developing Edge-AI Solutions for ST Platforms" activity so you can follow the exercises and reproduce the tests.

This script is created to test the trained model in a local environment.
  

**Application Steps:**

1. Include requirements

2. Load the trained model

3. Read a data sample from the serial port & perform inference using the trained model

## 1. Include Requirements

Imports for required modules:

In [ ]:
import tensorflow as tf
import serial
import numpy as np

## 2. Loading the Trained Model

Loading the trained model:

In [ ]:
import os
import re

def get_largest_performance_model(directory):
    pattern = re.compile(r"Model_CNN_Hand_Character_Recognition_0_(\d+)\.h5")

    max_performance = -1
    best_model = None

    for filename in os.listdir(directory):
        match = pattern.match(filename)
        if match:
            performance = int(match.group(1))
            if performance > max_performance:
                max_performance = performance
                best_model = filename

    return best_model

directory = "./Models"
name_best_model = get_largest_performance_model(directory)
print(f"The model with the best performance is: {name_best_model}")

In [ ]:
model = tf.keras.models.load_model(f"Models/{name_best_model}")

Defining the class mapping dictionary:

In [ ]:
# class ID to class name mapping
classes = {0: "CIRCLE",
           1: "HORIZONTAL",
           2: "STANDBY",
           3: "TRIANGLE",
           4: "VERTICAL"}

## 3. Read sample from Serial Port & Inference using the trained model

Define the serial port configuration values:

In [ ]:
serial_port = '/dev/ttyACM0'   # Ubuntu
# serial_port = 'COM0'             # Windows
baud_rate = 115200
time_out = 2

Read data from the serial port & perform inference on received samples

In [ ]:
# collect data, get inference from model
while True:
    # open the serial port, set baudrate, set timeout
    with serial.Serial(serial_port, baud_rate, timeout=time_out) as ser_read:
        # read the line from serial port
        x = ser_read.readline()
        # parse the line into numpy array
        line = np.array(str(x).replace("b'", "").replace("\\n'", "").replace("\\r", "").split(" ")[:-1])
        # ignore the line if the length is not 6*128=768
        if len(line) == 768:
            # convert the array of strings to array of floating point numbers
            line = line.astype(np.float32)
            # reshape the array for model
            line = line.reshape(128,6)
            # add a batch shape to the array
            line = np.expand_dims(line, axis=0)
            # get the model output
            out = model(line)
            # model outputs a tensor, convert it to numpy array
            out = np.array(out)
            # get the highest scoring output as predicted class
            class_index = np.argmax(out)
            # get the class name from dict
            class_name = classes[class_index]
            # print the results
            print(f"\rProbabilities -> CIRCLE: {out[0,0]:.2f} HORIZONTAL: {out[0,1]:.2f} STANDBY: {out[0,2]:.2f} TRIANGLE: {out[0,3]:.2f} VERTICAL: {out[0,4]:.2f} --- Prediction: {class_name} {' '*50}",end="")